In [ ]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"
system_prompt = """
あなたはとても簡潔にソリューションを提示できる優秀なエンジニアです。
"""

In [ ]:
# tool
from anthropic.types import ToolParam
from datetime import datetime, timedelta

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

def add_duration_to_datetime(datetime_str, days=0, hours=0, minutes=0, seconds=0, date_format="%Y-%m-%d %H:%M:%S"):
    dt = datetime.strptime(datetime_str, date_format)
    result = dt + timedelta(days=days, hours=hours, minutes=minutes, seconds=seconds)
    return result.strftime(date_format)

# 単にjsonオブジェクトを定義してもよいが、ToolParamでラップするとコードの堅牢性が上がるらしい(?)
get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "現在の日時を指定したフォーマットの文字列で取得します。",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "datetime.strftime に渡す日時フォーマット文字列",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []
    }
})

add_duration_to_datetime_schema = ToolParam({
    "name": "add_duration_to_datetime",
    "description": "指定した日時に期間(日数・時間・分・秒)を加算(または減算)した結果を文字列で取得します。",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "基準となる日時の文字列(date_formatに従う)"
            },
            "days": {
                "type": "integer",
                "description": "加算する日数(負の値で減算)",
                "default": 0
            },
            "hours": {
                "type": "integer",
                "description": "加算する時間数(負の値で減算)",
                "default": 0
            },
            "minutes": {
                "type": "integer",
                "description": "加算する分数(負の値で減算)",
                "default": 0
            },
            "seconds": {
                "type": "integer",
                "description": "加算する秒数(負の値で減算)",
                "default": 0
            },
            "date_format": {
                "type": "string",
                "description": "datetime のパース・フォーマットに使う日時フォーマット文字列",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": ["datetime_str"]
    }
})

def run_tool(tool_name, tool_input):
    # ツール名 -> 実行する関数 のマッピング
    tool_functions = {
        "get_current_datetime": get_current_datetime,
        "add_duration_to_datetime": add_duration_to_datetime,
    }
    return tool_functions[tool_name](**tool_input)
    

In [ ]:
# web_search はサーバーサイドツールなので input_schema は不要で、run_tool にも登録しない
# (Anthropic側で検索が実行され、結果は tool_use ではなく web_search_tool_result ブロックとしてレスポンスに含まれる)
web_search_tool_schema = ToolParam({
    # 動的フィルタリング版(web_search_20260209)は Opus 4.6以降/Sonnet 4.6以降 が対象で、
    # claude-haiku-4-5 は非対応のため、こちらの旧バージョンを使う
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,  # 1リクエストあたりの検索回数上限(任意)
})

In [ ]:
import json

# Claude メッセージヘルパー
def to_user_message(content):
    user_message = {"role": "user", "content": content}
    return user_message

def to_assistant_message(content):
    assistant_message = {"role": "assistant", "content": content}
    return assistant_message

def execute_tool_use_blocks(content_blocks):
    """response.content内のtool_useブロックを実行し、tool_resultブロックのリストを返す"""
    tool_results = []
    for block in content_blocks:
        if block.type == "tool_use":
            try :
                result = run_tool(block.name, block.input)
                isError = False
            except Exception as e:
                result = f"Error: {e}"
                isError = True
            
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": json.dumps(result),
                "is_error": isError
            })
    return tool_results

def chat(messages, temperature=0.0):
    return client.messages.stream(
        model=model,
        max_tokens=1000,
        temperature=temperature,
        messages=messages,
        system=system_prompt,
        tools=[get_current_datetime_schema, add_duration_to_datetime_schema, web_search_tool_schema]
    )

In [ ]:
messages = []
while(True):
    inputMessage = input("> ")
    if inputMessage == '':
        print('no text input, terminate this program.')
        break

    print(">", inputMessage)
    user_message = to_user_message(inputMessage)
    messages.append(user_message)

    print('--------- model answer')
    with chat(messages) as stream:
        for text in stream.text_stream:
            print(text, end="", flush=True)
        response = stream.get_final_message()
    print()
    print('---------')
    assistant_message = to_assistant_message(response.content)
    messages.append(assistant_message)

    # Claudeがツール利用を要求した場合、実行結果を返してから改めて応答を取得する
    while response.stop_reason == "tool_use":
        print('--------- model tool request')
        print(list(filter(lambda x: x.type == 'tool_use', response.content)))
        print('---------')
        tool_result_message = to_user_message(execute_tool_use_blocks(response.content))
        messages.append(tool_result_message)
        print('--------- system')
        print(tool_result_message)
        print('---------')

        with chat(messages) as stream:
            for text in stream.text_stream:
                print(text, end="", flush=True)
            response = stream.get_final_message()
        print()
        print('---------')
        assistant_message = to_assistant_message(response.content)
        messages.append(assistant_message)